# Aayush kumar Bohara (15938004)

## PySpark Lab Work: Exploratory Data Analysis (EDA) Using PySpark and Spark SQL

In [2]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create Spark session (if not already created)
spark = SparkSession.builder.appName("EDA").getOrCreate()

26/06/07 11:18:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### Part A: Data Loading and Inspection

In [3]:
df = spark.read.csv("supermarket_sales.csv", header = True, inferSchema=True )

df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)



In [6]:
df.show(10)

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|      date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|      city|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|         T0001|2025-02-05|       C027|       Beverages|      Coffee|       5|     13.38|       66.9|       eWallet|Biratnagar|
|         T0002|2025-02-25|       C060|       Beverages|  Soft Drink|       1|      2.95|       2.95|       eWallet|Biratnagar|
|         T0003|2025-04-25|       C072|       Beverages|         Tea|       9|     21.85|     196.65|          Cash|   Pokhara|
|         T0004|2025-03-28|       C027|          Snacks|   Chocolate|       5|      9.04|       45.2|          Cash| Bhaktapur|
|         T0005|2025-01-12|       C118|       Beverages|       Juice|       6|     30.78|     184.68|   

In [7]:
row_count = df.count()

column_count = len(df.columns)

print("Total No of rows: ", row_count)
print("Total No of columns: ", column_count)

Total No of rows:  500
Total No of columns:  10


In [8]:
from pyspark.sql.functions import col, sum as spark_sum, isnan, when

In [10]:
null_counts = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise (0)).alias(c)
    for c in df.columns
])

In [11]:
null_counts.show()

+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|             0|   0|          0|               0|           0|       0|         0|          0|             0|   0|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+



In [12]:
duplicate_count = df.groupby(df.columns).count().filter(col("count")> 1)

In [14]:
duplicate_count.show()

+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|count|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+



In [17]:
#Calcuates descriptive statistics for all numerical columns.

df.describe().show()

+-------+--------------+-----------+----------------+------------+------------------+------------------+-----------------+--------------+---------+
|summary|transaction_id|customer_id|product_category|product_name|          quantity|        unit_price|      total_sales|payment_method|     city|
+-------+--------------+-----------+----------------+------------+------------------+------------------+-----------------+--------------+---------+
|  count|           500|        500|             500|         500|               500|               500|              500|           500|      500|
|   mean|          NULL|       NULL|            NULL|        NULL|             5.524|25.628980000000013|        145.03284|          NULL|     NULL|
| stddev|          NULL|       NULL|            NULL|        NULL|2.9730627649628962|13.991613046066618|120.2808106872146|          NULL|     NULL|
|    min|         T0001|       C002|          Bakery|       Apple|                 1|              1.73|        

### Part B: EDA Using PySpark DataFrame Methods

In [19]:
df.select(spark_sum("total_sales")).show()

+----------------+
|sum(total_sales)|
+----------------+
|        72516.42|
+----------------+



In [22]:
df.groupBy("product_category").sum("total_sales").orderBy(col("sum(total_sales)").desc()).show(5)

+----------------+------------------+
|product_category|  sum(total_sales)|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+



In [23]:
from pyspark.sql.functions import avg

In [24]:
df.select(avg("quantity"), avg("unit_price"), avg("total_sales")).show()

+-------------+------------------+----------------+
|avg(quantity)|   avg(unit_price)|avg(total_sales)|
+-------------+------------------+----------------+
|        5.524|25.628980000000013|       145.03284|
+-------------+------------------+----------------+



In [25]:
df.groupBy("city").sum("total_sales").orderBy(col("sum(total_sales)").desc()).show(5)

+----------+------------------+
|      city|  sum(total_sales)|
+----------+------------------+
| Bhaktapur|16650.069999999996|
|  Lalitpur|16323.509999999998|
|   Pokhara|14540.320000000003|
| Kathmandu|          13034.83|
|Biratnagar|11967.690000000002|
+----------+------------------+



In [28]:
df.groupBy("payment_method").count().orderBy(col("count").desc()).show()

+--------------+-----+
|payment_method|count|
+--------------+-----+
|          Card|  177|
|          Cash|  163|
|       eWallet|  160|
+--------------+-----+



In [29]:
df.groupBy("product_category").sum("total_sales").orderBy(col("sum(total_sales)").desc()).show()

+----------------+------------------+
|product_category|  sum(total_sales)|
+----------------+------------------+
|          Snacks|15938.829999999998|
|           Dairy|15883.500000000004|
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|         Produce|12298.050000000001|
+----------------+------------------+



In [30]:
df.orderBy(col("total_sales").desc()).show(10)

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|      date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|      city|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|         T0015|2025-05-16|       C065|          Snacks|    Crackers|      10|     49.81|      498.1|       eWallet| Kathmandu|
|         T0093|2025-05-15|       C073|          Snacks|   Chocolate|      10|     48.87|      488.7|          Cash|   Pokhara|
|         T0100|2025-01-26|       C054|          Bakery|       Bread|      10|     48.07|      480.7|       eWallet|   Pokhara|
|         T0161|2025-05-24|       C085|           Dairy|        Milk|      10|      47.9|      479.0|       eWallet| Kathmandu|
|         T0098|2025-01-22|       C064|           Dairy|      Cheese|      10|     47.53|      475.3|   

In [31]:
df2 = df.withColumn(
    "sales_level",
    when(col("total_sales") > 100, "High").
    when((col("total_sales") >= 50 ) & (col("total_sales") <= 100), "Medium")
    .otherwise("Low")
    
)

In [32]:
df2.groupBy("sales_level").count().show()

+-----------+-----+
|sales_level|count|
+-----------+-----+
|       High|  261|
|        Low|  139|
|     Medium|  100|
+-----------+-----+



### Part C: EDA Using Spark SQL

In [34]:
df.createOrReplaceTempView("sales_view")

In [35]:
spark.sql("""
SELECT SUM(total_sales) AS total_revenue
FROM sales_view
""").show()

+-------------+
|total_revenue|
+-------------+
|     72516.42|
+-------------+



In [36]:
spark.sql("""
SELECT city, SUM(total_sales) AS revenue
FROM sales_view
GROUP BY city
""").show()

+----------+------------------+
|      city|           revenue|
+----------+------------------+
|   Pokhara|14540.320000000003|
|  Lalitpur|16323.509999999998|
| Kathmandu|          13034.83|
| Bhaktapur|16650.069999999996|
|Biratnagar|11967.690000000002|
+----------+------------------+



In [37]:
spark.sql("""
SELECT product_category, SUM(total_sales) AS revenue
FROM sales_view
GROUP BY product_category
""").show()

+----------------+------------------+
|product_category|           revenue|
+----------------+------------------+
|          Bakery|14209.069999999994|
|       Beverages|14186.970000000005|
|           Dairy|15883.500000000004|
|         Produce|12298.050000000001|
|          Snacks|15938.829999999998|
+----------------+------------------+



In [38]:
spark.sql("""
SELECT product_category, AVG(quantity) AS avg_quantity
FROM sales_view
GROUP BY product_category
""").show()

+----------------+------------------+
|product_category|      avg_quantity|
+----------------+------------------+
|          Bakery|5.6020408163265305|
|       Beverages| 5.245283018867925|
|           Dairy| 5.904255319148936|
|         Produce| 5.540816326530612|
|          Snacks|             5.375|
+----------------+------------------+



In [39]:
spark.sql("""
SELECT product_name, SUM(total_sales) AS total_sales
FROM sales_view
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 5
""").show()

+------------+------------------+
|product_name|       total_sales|
+------------+------------------+
|         Tea| 5265.120000000001|
|        Milk| 5056.429999999999|
|     Cookies| 4913.889999999999|
|       Apple|4596.6100000000015|
|      Yogurt| 4299.000000000001|
+------------+------------------+



In [40]:
spark.sql("""
SELECT payment_method, COUNT(*) AS transactions
FROM sales_view
GROUP BY payment_method
""").show()

+--------------+------------+
|payment_method|transactions|
+--------------+------------+
|          Card|         177|
|          Cash|         163|
|       eWallet|         160|
+--------------+------------+



In [41]:
spark.sql("""
SELECT MAX(total_sales) AS highest_transaction
FROM sales_view
""").show()

+-------------------+
|highest_transaction|
+-------------------+
|              498.1|
+-------------------+



In [42]:
spark.sql("""
SELECT date,
       SUM(total_sales) AS total_revenue,
       COUNT(*) AS transactions
FROM sales_view
GROUP BY date
ORDER BY date
""").show()
 

+----------+------------------+------------+
|      date|     total_revenue|transactions|
+----------+------------------+------------+
|2025-01-01|185.85000000000002|           2|
|2025-01-02|             223.3|           1|
|2025-01-03|            111.04|           1|
|2025-01-04|            688.55|           4|
|2025-01-05|             160.4|           1|
|2025-01-07|            132.75|           1|
|2025-01-08|            433.02|           2|
|2025-01-09|            828.78|           4|
|2025-01-10|392.63000000000005|           3|
|2025-01-11|              43.2|           1|
|2025-01-12|             526.3|           3|
|2025-01-13|            621.75|           3|
|2025-01-14| 897.1800000000001|           4|
|2025-01-15|            945.99|           5|
|2025-01-16|            658.35|           2|
|2025-01-17|            229.68|           2|
|2025-01-18| 852.9499999999999|           5|
|2025-01-19|            288.01|           3|
|2025-01-20|            580.51|           5|
|2025-01-2

### Part D: Insights and Interpretation

1.Beverages likely generate the highest revenue category.
2.Certain cities like Kathmandu/Pokhara dominate sales performance.
3.Cashless payments (eWallet/Card) are widely used.
4.A small number of high-value transactions contribute significantly to revenue.
5.Sales vary across dates, showing fluctuating daily demand